# Guía del pipeline de entrenamiento de Habla IA

## Qué es

Es la guía de los scripts con los que se entrena el modelo que usa la app para distinguir el español del maya, reúne los cinco scripts de la versión con la que se entrenó el modelo actual en el mismo orden en que se usaron, explica qué hace cada uno y cómo se corre por separado, y además permite ejecutarlos de principio a fin desde aquí sin tener que abrirlos uno por uno.

## Qué hace

Parte de los audios originales de cada idioma, les quita los silencios y los corta en clips de cinco segundos con `PreparadorAudios.py`, los pasa a 16 kHz en mono con `hacer16000.py`, convierte cada clip en una imagen de espectrograma con el generador de Kotlin de la carpeta `untitled`, entrena con esas imágenes una red MobileNetV3Large usando `entrenador.py`, evalúa el modelo ya convertido a TFLite como lo hace `Untitled1_corregido.ipynb` y al final deja listo lo necesario para llevar el modelo a la app.

Es la única versión que queda en `Entrenamiento`, las anteriores se quitaron del proyecto y de ellas solo se conservan en `Modulos no usados/cambiador` algunos scripts viejos, que se describen al final de esta guía.

## Cómo se usa

Primero se revisan los requisitos y se ajustan las rutas en la celda de configuración, después se ejecutan las secciones en orden porque cada paso lee lo que dejó el anterior, y si se prefiere correr algún script por fuera del notebook cada sección explica qué líneas hay que cambiar y qué comando usar.

## Requisitos, lo que hay que tener instalado

### Qué hace

Revisa que estén instaladas las librerías de Python que usa el proceso, que son TensorFlow para entrenar, NumPy, SciPy y Pillow para los espectrogramas, pydub para cortar y convertir el audio, scikit-learn, pandas y openpyxl para las métricas y el Excel, y matplotlib para las gráficas.

### Cómo se usa

Se ejecuta la celda y muestra cuáles faltan, si falta alguna se quita el `#` de la línea de `%pip install` y se vuelve a ejecutar. TensorFlow todavía no funciona en Python 3.14, por eso conviene usar Python 3.10, 3.11 o 3.12, y en Python 3.13 o más nuevo pydub necesita además `audioop-lts` porque usa un módulo que se quitó de Python. Como aquí pydub solo lee y escribe WAV no necesita ffmpeg, así que el aviso de que no lo encuentra se puede ignorar. Para generar los espectrogramas con el proyecto original de Kotlin hace falta el JDK 23 e IntelliJ IDEA, aunque el notebook trae una versión en Python del mismo algoritmo.

In [ ]:
# Quita el # de estas líneas si falta algo
# %pip install tensorflow numpy scipy pillow pydub scikit-learn pandas openpyxl matplotlib
# %pip install audioop-lts

import importlib

# Intenta importar cada librería y muestra su versión, o avisa si no está
for paquete in ["numpy", "scipy", "PIL", "pydub", "sklearn", "pandas", "openpyxl", "matplotlib", "tensorflow"]:
    try:
        modulo = importlib.import_module(paquete)
        print(f"✔ {paquete:<11} {getattr(modulo, '__version__', '')}")
    except ImportError:
        print(f"✘ {paquete:<11} FALTA")

## Configuración, las rutas y parámetros de todo el proceso

### Qué hace

Reúne en una sola celda las carpetas de entrada y salida de cada paso, los nombres de los modelos y los parámetros de audio, que son los mismos que usa la app, clips de cinco segundos a 16 000 Hz. Los scripts originales tienen cada uno sus rutas escritas dentro del código y no coinciden entre sí, uno escribe en `entrenamien` y el siguiente lee de `entrenamiento`, y el entrenador guarda `modelo_maya_español` con ñ mientras la evaluación busca `modelo_maya_espanol` sin ñ, por eso aquí se definen una sola vez.

### Cómo se usa

Se pone en `AUDIOS_ORIGINALES` la carpeta con los audios, con una subcarpeta por idioma como `audios_originales/español` y `audios_originales/maya`, y se ejecuta la celda, que indica si encontró los audios, el generador de Kotlin y la carpeta de modelos de la app. El nombre de cada subcarpeta es la etiqueta del idioma y el orden alfabético decide qué número le toca a cada uno, así que `español` es la clase 0 y `maya` la clase 1.

In [ ]:
from pathlib import Path

# Carpeta Entrenamiento, donde está este notebook
BASE = Path.cwd()

# Audios originales, una subcarpeta por idioma como audios_originales/español y audios_originales/maya
AUDIOS_ORIGINALES = BASE / "audios_originales"

# Carpeta que deja cada paso, la del paso 4 se borra y se vuelve a crear en cada corrida
CLIPS_5S          = BASE / "entrenamien"
CLIPS_16K         = BASE / "entrenamiento"
ESPECTROGRAMAS    = BASE / "espectrogramas"
SPLIT             = BASE / "dataset_split"
MODELO_KERAS      = BASE / "modelo_maya_espanol.keras"
MODELO_TFLITE     = BASE / "modelo_maya_espanol.tflite"

# Duración de cada clip y frecuencia de muestreo, iguales a las de la app
DURACION_CLIP_MS  = 5000
SAMPLE_RATE       = 16000

# Generador de espectrogramas original y carpeta de modelos de la app, que está en la raíz del repositorio
KOTLIN_GENERADOR  = BASE / "untitled" / "src" / "main" / "kotlin" / "MelSpectogramGenerator.kt"
APP_MODELOS       = BASE.parent / "lib" / "modelos"

for nombre, ruta in [("Audios originales", AUDIOS_ORIGINALES), ("Generador Kotlin", KOTLIN_GENERADOR), ("App", APP_MODELOS)]:
    print(f"{'✔' if ruta.exists() else '✘ no existe'}  {nombre}: {ruta}")

## Paso 1 · `PreparadorAudios.py`, el script que limpia y corta los audios

### Qué hace

Abre cada WAV, calcula su volumen promedio y toma como silencio todo lo que quede 20 dB por debajo, de modo que el umbral se adapta igual a audios grabados fuerte que bajito, quita los tramos de silencio de al menos 300 ms dejando 100 ms de margen para no cortar palabras, vuelve a unir los pedazos con voz y corta el resultado en clips de exactamente cinco segundos, y si el último queda más corto lo completa con silencio. Cada clip se guarda como `<nombre>_clip_<n>.wav`. La app no quita silencios sino que clasifica los cinco segundos tal como se grabaron, una diferencia que se explica en el paso 6.

### Cómo se usa

El script original procesa un solo idioma por corrida, se cambian al final de `PreparadorAudios.py` la `CARPETA_ENTRADA` con los audios de un idioma y la `CARPETA_SALIDA` donde van los clips, por ejemplo `audios/mayan/original` y `entrenamien/maya`, se ejecuta `python PreparadorAudios.py` y se repite con cada idioma. La celda de abajo usa la misma función pero procesa todos los idiomas de `AUDIOS_ORIGINALES` de una vez y deja los clips en `CLIPS_5S`.

In [ ]:
from pydub import AudioSegment
from pydub.silence import split_on_silence


# Misma función que PreparadorAudios.py, solo que devuelve cuántos clips generó
def procesar_audio_completo(ruta_input, ruta_output_folder, clip_length_ms=5000):
    audio = AudioSegment.from_wav(ruta_input)
    filename = Path(ruta_input).stem

    # Todo lo que esté 20 dB por debajo del volumen promedio cuenta como silencio
    umbral_dinamico = audio.dBFS - 20

    chunks = split_on_silence(
        audio,
        min_silence_len=300,
        silence_thresh=umbral_dinamico,
        keep_silence=100,
    )
    audio_limpio = sum(chunks) if chunks else audio

    # Corta en bloques de cinco segundos y rellena con silencio el último si queda corto
    total_duration = len(audio_limpio)
    start_time, clip_index = 0, 1
    while start_time < total_duration:
        clip = audio_limpio[start_time:start_time + clip_length_ms]
        if len(clip) < clip_length_ms:
            clip = clip + AudioSegment.silent(duration=clip_length_ms - len(clip))
        clip.export(Path(ruta_output_folder) / f"{filename}_clip_{clip_index}.wav", format="wav")
        start_time += clip_length_ms
        clip_index += 1
    return clip_index - 1


# Recorre cada carpeta de idioma y deja sus clips en una subcarpeta con el mismo nombre
def paso1_preparar_todos(entrada=AUDIOS_ORIGINALES, salida=CLIPS_5S):
    for carpeta_idioma in sorted(p for p in entrada.iterdir() if p.is_dir()):
        destino = salida / carpeta_idioma.name
        destino.mkdir(parents=True, exist_ok=True)
        wavs = sorted(p for p in carpeta_idioma.iterdir() if p.suffix.lower() == ".wav")
        total = sum(procesar_audio_completo(w, destino, DURACION_CLIP_MS) for w in wavs)
        print(f"{carpeta_idioma.name:>10}: {len(wavs)} audios → {total} clips de 5 s")


paso1_preparar_todos()

## Paso 2 · `hacer16000.py`, el script que pasa el audio a 16 kHz mono

### Qué hace

Abre cada clip, lo convierte a 16 000 Hz y a un solo canal con pydub y lo guarda con el mismo nombre en otra carpeta, así cada clip queda con unas 80 000 muestras. Aunque el generador de espectrogramas también baja la frecuencia a 16 kHz, este paso sigue haciendo falta porque el generador solo lee el canal izquierdo de un audio estéreo y usa una interpolación lineal simple, y con los clips ya en 16 kHz mono se evitan las dos cosas.

### Cómo se usa

Igual que el anterior, el script original trabaja un idioma a la vez, se cambian `CARPETA_ENTRADA` y `CARPETA_SALIDA`, por ejemplo a `entrenamien/español` y `entrenamiento/español`, y se ejecuta `python hacer16000.py` para cada idioma. La celda de abajo lo hace para todos, leyendo de `CLIPS_5S` y escribiendo en `CLIPS_16K`.

In [ ]:
# Convierte cada clip a 16 kHz mono y lo guarda con el mismo nombre en CLIPS_16K
def paso2_a_16k(entrada=CLIPS_5S, salida=CLIPS_16K, sample_rate=SAMPLE_RATE):
    for carpeta_idioma in sorted(p for p in entrada.iterdir() if p.is_dir()):
        destino = salida / carpeta_idioma.name
        destino.mkdir(parents=True, exist_ok=True)
        n = 0
        for wav in sorted(carpeta_idioma.glob("*.wav")):
            audio = AudioSegment.from_wav(wav).set_frame_rate(sample_rate).set_channels(1)
            audio.export(destino / wav.name, format="wav")
            n += 1
        print(f"{carpeta_idioma.name:>10}: {n} clips a {sample_rate} Hz mono")


paso2_a_16k()

## Paso 3 · Generador de Kotlin en `untitled/`, el programa que convierte cada clip en un espectrograma

### Qué hace

Lee cada clip, lo divide en ventanas de 2048 muestras que avanzan de 512 en 512, calcula la FFT de cada ventana, agrupa las frecuencias en 128 bandas Mel entre 0 y 8000 Hz, pasa la energía a decibeles tomando como referencia el punto más fuerte del clip con un piso de −80 dB y dibuja el resultado en una imagen de 224 por 224 con la paleta de colores inferno, con los graves abajo y el tiempo de izquierda a derecha. Es el paso que más importa que coincida con la app, porque el modelo aprende de estas imágenes y en el teléfono `GeneradorEspectrograma.kt` dibuja la suya con el mismo algoritmo y los mismos valores, que en la app están en `ParametrosEspectrograma` como `sampleRate: 16000`, `nFft: 2048`, `hopLength: 512`, `nMels: 128`, `fMin: 0`, `fMax: 8000`, `dbMin: -80` y `MapaColor.inferno`. La paleta tiene 256 colores aunque el comentario del `.kt` diga 252, y es idéntica color por color a la de la app.

### Cómo se usa

Se abre la carpeta `untitled/` en IntelliJ IDEA como proyecto Gradle con el JDK 23 y se ejecuta `main()` de `src/main/kotlin/Main.kt` con el botón que aparece junto a la función, que lee los clips de `entrenamiento/`, la carpeta que deja el paso 2, y crea `espectrogramas/<idioma>/<clip>.png`, que es donde los busca el paso 4. Las dos rutas están en `CARPETA_ENTRADA` y `CARPETA_SALIDA` y son relativas a `untitled/`, que es la carpeta desde donde IntelliJ ejecuta el proyecto, así que funcionan en cualquier computadora sin cambiarlas, y al empezar el programa muestra la ruta completa de cada una para confirmar que son las correctas. `MODO_SUBCARPETAS` se deja en `true` para que respete una subcarpeta por idioma.

## Versión en Python del generador, por si no se tiene Kotlin

### Qué hace

Es una traducción línea por línea de `MelSpectogramGenerator.kt`, toma la paleta directamente del archivo `.kt` para que sea la misma y hace en `float32` las cuentas que deciden a qué píxel va cada valor, igual que Kotlin, de modo que la única diferencia posible es el orden en que se suman los números, que podría mover algún píxel suelto a un color vecino.

### Cómo se usa

Se deja `USAR_PYTHON_PARA_ESPECTROGRAMAS` en `True` y se ejecuta la celda, que lee de `CLIPS_16K` y escribe en `ESPECTROGRAMAS`, y si los PNG ya se hicieron con Kotlin se cambia a `False` para no sobrescribirlos.

In [ ]:
import re
import numpy as np
from PIL import Image
from scipy.io import wavfile

# Mismos valores que MelSpectogramGenerator.kt y que la app
N_FFT, HOP_LENGTH, N_MEL = 2048, 512, 128
F_MIN, F_MAX, DB_MIN, OUTPUT_SIZE = 0.0, 8000.0, np.float32(-80.0), 224


# Lee la paleta inferno del propio archivo .kt para usar exactamente los mismos colores
def leer_lut_inferno(ruta_kt=KOTLIN_GENERADOR):
    texto = Path(ruta_kt).read_text(encoding="utf-8")
    colores = re.findall(r"intArrayOf\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)", texto)
    return np.array(colores, dtype=np.uint8)


INFERNO_LUT = leer_lut_inferno()
print(f"Tabla inferno: {len(INFERNO_LUT)} colores")


# Equivale a cargarWav, toma el canal izquierdo, lo pasa a valores entre -1 y 1 y lo lleva a 16 kHz
def cargar_wav(ruta):
    sr, datos = wavfile.read(ruta)
    if datos.ndim > 1:
        datos = datos[:, 0]
    if datos.dtype == np.int16:
        x = datos.astype(np.float32) / 32768.0
    elif datos.dtype == np.int32:
        x = datos.astype(np.float32) / 2147483648.0
    elif datos.dtype == np.float32:
        x = np.clip(datos, -1.0, 1.0)
    else:
        raise ValueError(f"Formato WAV no soportado: {datos.dtype}")
    return resamplear(x, sr, SAMPLE_RATE)


# Cambia la frecuencia de muestreo con interpolación lineal, igual que resamplear en Kotlin
def resamplear(x, de, a):
    if de == a:
        return x
    ratio = de / a
    n = int(len(x) / ratio)
    pos = np.arange(n) * ratio
    idx = np.clip(pos.astype(np.int64), 0, len(x) - 2)
    frac = (pos - idx).astype(np.float32)
    return x[idx] * (1 - frac) + x[idx + 1] * frac


# Arma los 128 filtros triangulares que agrupan las frecuencias de la FFT en bandas Mel
def banco_mel(num_bins):
    hz_a_mel = lambda hz: 2595.0 * np.log10(1.0 + hz / 700.0)
    mel_a_hz = lambda mel: 700.0 * (10.0 ** (mel / 2595.0) - 1.0)
    mel_min, mel_max = hz_a_mel(F_MIN), hz_a_mel(F_MAX)
    puntos = mel_a_hz(mel_min + np.arange(N_MEL + 2) * (mel_max - mel_min) / (N_MEL + 1))
    bins = np.floor(puntos / (SAMPLE_RATE / N_FFT))
    k = np.arange(num_bins, dtype=np.float64)
    banco = np.zeros((N_MEL, num_bins), dtype=np.float32)
    for m in range(N_MEL):
        izq, centro, der = bins[m], bins[m + 1], bins[m + 2]
        ancho_izq, ancho_der = centro - izq, der - centro
        subida = (k >= izq) & (k <= centro)
        bajada = (k > centro) & (k <= der)
        banco[m, subida] = (k[subida] - izq) / ancho_izq if ancho_izq > 0 else 1.0
        banco[m, bajada] = (der - k[bajada]) / ancho_der if ancho_der > 0 else 1.0
    return banco


# Equivale a generarImagen, calcula la STFT, la pasa a bandas Mel en decibeles y la pinta en 224 por 224
def generar_imagen(muestras):
    ventana = np.hanning(N_FFT).astype(np.float32)
    num_frames = max(1, (len(muestras) - N_FFT) // HOP_LENGTH + 1)
    relleno = np.zeros(max(len(muestras), (num_frames - 1) * HOP_LENGTH + N_FFT), dtype=np.float32)
    relleno[:len(muestras)] = muestras
    tramas = np.stack([relleno[f * HOP_LENGTH:f * HOP_LENGTH + N_FFT] * ventana for f in range(num_frames)])
    magnitudes = np.abs(np.fft.rfft(tramas, axis=1)).astype(np.float32).T

    energia = banco_mel(magnitudes.shape[0]) @ magnitudes
    mel_db = (20.0 * np.log10(np.maximum(energia, np.float32(1e-10)))).astype(np.float32)
    max_db = mel_db.max()

    # Elige qué banda y qué ventana le tocan a cada píxel haciendo las cuentas en float32 como Kotlin
    t = np.float32(OUTPUT_SIZE)
    filas = ((np.float32(OUTPUT_SIZE - 1) - np.arange(OUTPUT_SIZE, dtype=np.float32)) / t * np.float32(N_MEL)).astype(np.int64)
    columnas = (np.arange(OUTPUT_SIZE, dtype=np.float32) / t * np.float32(num_frames)).astype(np.int64)
    filas = np.clip(filas, 0, N_MEL - 1)
    columnas = np.clip(columnas, 0, num_frames - 1)

    # Normaliza entre el piso de -80 dB y el máximo del clip y lo convierte en un color de la paleta
    valores = mel_db[filas][:, columnas]
    norm = (np.maximum(valores - max_db, DB_MIN) - DB_MIN) / (-DB_MIN)
    indices = np.clip((norm * np.float32(len(INFERNO_LUT) - 1)).astype(np.int64), 0, len(INFERNO_LUT) - 1)
    return INFERNO_LUT[indices]


# Genera un PNG por cada clip, con una subcarpeta por idioma
def paso3_espectrogramas(entrada=CLIPS_16K, salida=ESPECTROGRAMAS):
    for carpeta_idioma in sorted(p for p in entrada.iterdir() if p.is_dir()):
        destino = salida / carpeta_idioma.name
        destino.mkdir(parents=True, exist_ok=True)
        wavs = sorted(carpeta_idioma.glob("*.wav"))
        for wav in wavs:
            Image.fromarray(generar_imagen(cargar_wav(wav))).save(destino / f"{wav.stem}.png")
        print(f"{carpeta_idioma.name:>10}: {len(wavs)} espectrogramas")


# Cámbialo a False si los PNG ya se generaron con el proyecto de Kotlin
USAR_PYTHON_PARA_ESPECTROGRAMAS = True
if USAR_PYTHON_PARA_ESPECTROGRAMAS:
    paso3_espectrogramas()

## Comparación con el generador de Kotlin, una prueba opcional

### Qué hace

Comprueba que la versión en Python y la de Kotlin dan la misma imagen, generando el espectrograma de un WAV en Python y comparándolo con el PNG que hizo Kotlin con ese mismo WAV, y muestra qué porcentaje de píxeles coincide y cuál es la diferencia media.

### Cómo se usa

Se quita el `#` de la última línea, se ponen la ruta de un WAV y la del PNG que generó Kotlin para él y se ejecuta, lo esperado es que coincidan casi todos los píxeles y que la diferencia media sea cercana a cero, y si no es así alguno de los dos generadores cambió y conviene quedarse con el de Kotlin.

In [ ]:
# Compara píxel por píxel la imagen de Python con la que generó Kotlin para el mismo WAV
def comparar_con_kotlin(ruta_wav, ruta_png_kotlin):
    python_img = generar_imagen(cargar_wav(ruta_wav)).astype(np.int16)
    kotlin_img = np.asarray(Image.open(ruta_png_kotlin).convert("RGB")).astype(np.int16)
    iguales = np.all(python_img == kotlin_img, axis=2).mean() * 100
    print(f"Píxeles idénticos: {iguales:.2f} %  |  diferencia media: {np.abs(python_img - kotlin_img).mean():.3f}")


# comparar_con_kotlin("entrenamiento/maya/audio1_clip_1.wav", "espectrogramas_kotlin/maya/audio1_clip_1.png")

In [ ]:
import matplotlib.pyplot as plt

# Muestra dos espectrogramas de cada idioma para revisarlos a simple vista
fig, ejes = plt.subplots(1, 4, figsize=(14, 4))
ejemplos = [png for carpeta in sorted(ESPECTROGRAMAS.iterdir()) if carpeta.is_dir()
            for png in sorted(carpeta.glob("*.png"))[:2]]
for eje, png in zip(ejes, ejemplos):
    eje.imshow(Image.open(png))
    eje.set_title(f"{png.parent.name}\n{png.name}", fontsize=9)
    eje.axis("off")
plt.tight_layout()

## Paso 4 · `entrenador.py`, el script que entrena el modelo y lo exporta a TFLite

### Qué hace

Primero reparte las imágenes en entrenamiento, validación y prueba en proporción 80, 10 y 10, tomando como máximo 2086 imágenes por idioma elegidas al azar con la semilla 42 para que los idiomas queden parejos, y las copia a `dataset_split/train`, `val` y `test`, borrando antes esa carpeta completa. Luego carga las imágenes dividiendo cada píxel entre 255, sin el preprocesamiento de ImageNet porque los espectrogramas no se parecen a fotos, y arma el modelo con una MobileNetV3Large preentrenada sin su capa final, a la que le agrega un promedio global, una normalización por lotes, un dropout de 0.3 y una sola neurona sigmoide. El entrenamiento va en dos fases, en la primera la base queda congelada y solo se entrena la parte nueva con un learning rate de 0.001 hasta por 20 épocas, y en la segunda se liberan las últimas 30 capas de la base y se ajustan con un learning rate de 0.00001 hasta por otras 20 épocas, las dos con parada temprana si la pérdida de validación no mejora en cinco épocas, con el learning rate reducido a la mitad tras tres épocas sin mejora y guardando siempre el mejor punto. Al final evalúa con el conjunto de prueba, guarda el `.keras` y exporta un TFLite en float16 de unos 6 MB.

Como las clases se ordenan alfabéticamente quedan como `español` y `maya`, así que la única neurona de salida da la probabilidad de que el audio sea maya, y por eso en la app el modelo se declara con `ConfiguracionSalida.binaria(etiquetas: ['español', 'maya'])`.

### Cómo se usa

Fuera del notebook basta con ejecutar `python entrenador.py` en esta carpeta, que lee los espectrogramas de `espectrogramas/` y escribe `dataset_split/`, los checkpoints de cada fase, `modelo_maya_español.keras` y `modelo_maya_español.tflite`, y como la evaluación busca el nombre sin ñ hay que renombrarlo. Aquí no se copia su código sino que se importa el propio `entrenador.py`, que tiene su flujo principal protegido con `if __name__ == "__main__"`, se le pasan las rutas de la configuración y se llama a sus funciones en el mismo orden, con la preparación de datos, las dos fases y la evaluación en celdas separadas para poder revisar cada parte.

In [ ]:
import os
import importlib
import tensorflow as tf
import entrenador

importlib.reload(entrenador)

# Las funciones de entrenador.py leen estas constantes del propio módulo
entrenador.IMG_SIZE      = 224
entrenador.BATCH_SIZE    = 32
entrenador.MAX_POR_CLASE = 2086
entrenador.EPOCHS_FASE1  = 20
entrenador.EPOCHS_FASE2  = 20
entrenador.SEED          = 42

print(f"TensorFlow {tf.__version__} | GPU: {tf.config.list_physical_devices('GPU') or 'no'}")

# Reparte las imágenes en train, val y test y muestra el orden de las clases
clases = entrenador.crear_split(str(ESPECTROGRAMAS), str(SPLIT), entrenador.MAX_POR_CLASE, entrenador.SEED)
print(f"\nOrden de clases (índice = posición): {clases}")

ds_train = entrenador.crear_dataset(str(SPLIT / "train"), entrenador.BATCH_SIZE, shuffle=True)
ds_val   = entrenador.crear_dataset(str(SPLIT / "val"),   entrenador.BATCH_SIZE, shuffle=False)
ds_test  = entrenador.crear_dataset(str(SPLIT / "test"),  entrenador.BATCH_SIZE, shuffle=False)

In [ ]:
modelo, base = entrenador.construir_modelo()

# Fase 1, con la base congelada solo aprende la parte nueva del modelo
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
historia1 = modelo.fit(ds_train, validation_data=ds_val, epochs=entrenador.EPOCHS_FASE1,
                       callbacks=entrenador.callbacks("checkpoint_fase1.keras", patience=5))

# Fase 2, se liberan las últimas 30 capas de la base y se ajustan con un learning rate mucho menor
base.trainable = True
for capa in base.layers[:-30]:
    capa.trainable = False
print(f"Capas entrenables en la base: {sum(c.trainable for c in base.layers)} / {len(base.layers)}")

modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=["accuracy"])
historia2 = modelo.fit(ds_train, validation_data=ds_val, epochs=entrenador.EPOCHS_FASE2,
                       callbacks=entrenador.callbacks("checkpoint_fase2.keras", patience=5))

In [ ]:
# Dibuja la accuracy de las dos fases seguidas, con una línea donde empieza el fine-tuning
acc     = historia1.history["accuracy"]     + historia2.history["accuracy"]
val_acc = historia1.history["val_accuracy"] + historia2.history["val_accuracy"]
plt.figure(figsize=(8, 4))
plt.plot(acc, label="train")
plt.plot(val_acc, ":", label="validación")
plt.axvline(len(historia1.history["accuracy"]) - 0.5, color="gray", lw=0.8)
plt.text(len(historia1.history["accuracy"]), min(acc + val_acc), " fine-tuning", color="gray")
plt.xlabel("época"); plt.ylabel("accuracy"); plt.legend(); plt.show()

# Evalúa con el conjunto de prueba, guarda el modelo y lo exporta a TFLite en float16
loss, acc_test = modelo.evaluate(ds_test, verbose=0)
print(f"Test → loss {loss:.4f} | accuracy {acc_test * 100:.2f} %")

modelo.save(MODELO_KERAS)
entrenador.exportar_tflite(modelo, str(MODELO_TFLITE))

## Paso 5 · `Untitled1_corregido.ipynb`, el notebook que evalúa el modelo convertido

### Qué hace

Comprueba que el archivo que de verdad corre en el teléfono, el TFLite en float16, sigue acertando igual que el modelo de Keras después de la conversión. Carga las imágenes de prueba con la misma división entre 255 del entrenamiento, pasa cada una por el intérprete de TFLite como lo hace la app, toma como maya todo lo que tenga una salida mayor a 0.5, ya que hay una sola neurona y no tiene sentido buscar el máximo, y calcula accuracy, precision, recall y F1 ponderados, la matriz de confusión y el reporte por clase, que guarda en `model_comparison.xlsx` con una hoja para la matriz, otra para el reporte y una de resumen.

### Cómo se usa

Se ejecuta la celda después del paso 4, lee `dataset_split/test` y el modelo de `MODELO_TFLITE`, muestra las métricas y la matriz de confusión y deja el Excel en esta carpeta.

In [ ]:
import time
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

# Carga las imágenes de prueba con la misma normalización del entrenamiento
test_ds = tf.keras.utils.image_dataset_from_directory(
    str(SPLIT / "test"), image_size=(224, 224), batch_size=32, shuffle=False, label_mode="binary")
nombres_clases = test_ds.class_names
test_ds = test_ds.map(lambda x, y: (x / 255.0, y))

X_test = np.concatenate([x.numpy() for x, _ in test_ds])
y_true = np.concatenate([y.numpy() for _, y in test_ds]).flatten().astype(int)

interprete = tf.lite.Interpreter(model_path=str(MODELO_TFLITE))
interprete.allocate_tensors()
entrada, salida = interprete.get_input_details()[0], interprete.get_output_details()[0]
print(f"Entrada {entrada['shape']} {entrada['dtype'].__name__} → salida {salida['shape']} {salida['dtype'].__name__}")

# Pasa las imágenes una por una por el intérprete, igual que la app
inicio = time.time()
puntajes = []
for imagen in X_test:
    interprete.set_tensor(entrada["index"], imagen[None].astype(entrada["dtype"]))
    interprete.invoke()
    puntajes.append(interprete.get_tensor(salida["index"])[0][0])
tiempo = time.time() - inicio

# Con una sola neurona sigmoide se usa un umbral de 0.5 en lugar de buscar el máximo
y_pred = (np.array(puntajes) > 0.5).astype(int)
metricas = {
    "Modelo": MODELO_TFLITE.stem,
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
    "Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
    "F1 Score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    "Inferencia (s)": tiempo,
    "Imágenes test": len(y_true),
    "Clases": len(nombres_clases),
}
for k, v in metricas.items():
    print(f"{k:<15} {v:.4f}" if isinstance(v, float) else f"{k:<15} {v}")

# Guarda la matriz de confusión, el reporte por clase y el resumen en el Excel
matriz = confusion_matrix(y_true, y_pred)
with pd.ExcelWriter(BASE / "model_comparison.xlsx", engine="openpyxl") as excel:
    pd.DataFrame(matriz, index=nombres_clases, columns=nombres_clases).to_excel(excel, sheet_name=f"{MODELO_TFLITE.stem}_CM"[:31])
    pd.DataFrame(classification_report(y_true, y_pred, target_names=nombres_clases, output_dict=True, zero_division=0)
                 ).transpose().to_excel(excel, sheet_name=f"{MODELO_TFLITE.stem}_Report"[:31])
    pd.DataFrame([metricas]).to_excel(excel, sheet_name="Summary", index=False)
print("\n✔ model_comparison.xlsx")
pd.DataFrame(matriz, index=[f"real {c}" for c in nombres_clases], columns=[f"pred {c}" for c in nombres_clases])

## Predicción de un solo espectrograma o de un audio

### Qué hace

`predecir_png` hace lo mismo que la función `predict_spectrogram_tflite` de `Untitled1_corregido.ipynb` y que la app, abre la imagen, la divide entre 255, la pasa por el modelo y, si la salida es mayor a 0.5, responde maya con esa probabilidad como confianza y si no responde español con la probabilidad contraria. `predecir_wav` agrega antes el paso de convertir un WAV de cinco segundos en espectrograma con el mismo algoritmo, lo que sirve para probar el modelo con los fragmentos que guarda la app en `fragmentos/bloque_N.wav`.

### Cómo se usa

Se llama a `predecir_png` con la ruta de un PNG o a `predecir_wav` con la de un WAV, la primera celda prueba una imagen de cada idioma del conjunto de prueba y la segunda trae un ejemplo comentado para un fragmento de la app.

In [ ]:
# Predice el idioma de un espectrograma y lo muestra con su confianza y la salida cruda del modelo
def predecir_png(ruta_png, ruta_tflite=MODELO_TFLITE, clases=("español", "maya")):
    interprete = tf.lite.Interpreter(model_path=str(ruta_tflite))
    interprete.allocate_tensors()
    entrada, salida = interprete.get_input_details()[0], interprete.get_output_details()[0]

    imagen = np.asarray(Image.open(ruta_png).convert("RGB").resize((224, 224)), dtype=np.float32) / 255.0
    interprete.set_tensor(entrada["index"], imagen[None].astype(entrada["dtype"]))
    interprete.invoke()
    p = float(interprete.get_tensor(salida["index"])[0][0])

    indice = int(p > 0.5)
    confianza = (p if indice == 1 else 1 - p) * 100
    print(f"{Path(ruta_png).name}: {clases[indice]} ({confianza:.1f} %)  · sigmoide = {p:.4f}")
    return clases[indice], confianza


for carpeta in sorted((SPLIT / "test").iterdir()):
    predecir_png(next(carpeta.glob("*.png")))

In [ ]:
# Convierte un WAV de cinco segundos en espectrograma con el mismo algoritmo de la app y lo clasifica
def predecir_wav(ruta_wav, ruta_tflite=MODELO_TFLITE, clases=("español", "maya")):
    temporal = BASE / "_prediccion_temporal.png"
    Image.fromarray(generar_imagen(cargar_wav(ruta_wav))).save(temporal)
    try:
        return predecir_png(temporal, ruta_tflite, clases)
    finally:
        temporal.unlink(missing_ok=True)


# Para probar con un fragmento grabado por la app, que viene en fragmentos/bloque_N.wav a 16 kHz
# predecir_wav("bloque_1.wav")

## Registro de experimentos, el historial en `log.csv`

### Qué hace

Anota cada entrenamiento en `log.csv` con los idiomas, el modelo base, la duración de los clips, cuántas imágenes hubo por idioma, los parámetros, la semilla, si los audios tenían silencio y los resultados, junto con un hash calculado a partir de todo eso que permite saber si una combinación ya se había probado. Usa las mismas columnas que ya tiene el archivo, así que cada corrida nueva queda junto a las anteriores.

### Cómo se usa

Se ejecuta después de la evaluación, toma los valores de la configuración y las métricas del paso 5, avisa si ya había un experimento con el mismo hash y agrega una fila nueva al archivo, dejando vacía la columna `Notes` para escribir observaciones a mano.

In [ ]:
import hashlib

# Datos del experimento, PreparadorAudios.py quita silencios y por eso con_silencio es False
idiomas      = sorted(nombres_clases, key=str.lower)
modelo_base  = "MobileNetV3Large"
imagenes     = {c: len(list((ESPECTROGRAMAS / c).glob("*.png"))) for c in idiomas}
parametros   = {"batch_size": entrenador.BATCH_SIZE, "frequency": SAMPLE_RATE,
                "lr_fase1": 1e-3, "lr_fase2": 1e-5, "max_por_clase": entrenador.MAX_POR_CLASE}
semilla      = entrenador.SEED
con_silencio = False

# El hash resume todos los parámetros, si se repite es que esa combinación ya se probó
hash_exp = hashlib.md5(
    f"{idiomas},{modelo_base},{DURACION_CLIP_MS // 1000},{imagenes}.{parametros},{semilla},{con_silencio}".encode()
).hexdigest()

ruta_log = BASE / "log.csv"
log = pd.read_csv(ruta_log) if ruta_log.exists() else pd.DataFrame()
if "Hash" in log and hash_exp in set(log["Hash"]):
    print(f"⚠️ Ya hay un experimento con estos parámetros: {hash_exp}")

nuevo = {
    "ID_Exp": f"EXP{len(log)}", "Hash": hash_exp, "Languages": idiomas, "Model": modelo_base,
    "Audio_Length": DURACION_CLIP_MS // 1000, "Image_Amount": imagenes, "Parameters": parametros,
    "Split_Seed": semilla, "hasSilence": con_silencio,
    "Results": (f"Accuracy: {metricas['Accuracy']:.4f} Recall: {metricas['Recall']:.4f} "
                f"Precision: {metricas['Precision']:.4f} F1: {metricas['F1 Score']:.4f}"),
    "Audio_Folders_Used": [str(AUDIOS_ORIGINALES / c) for c in idiomas], "Notes": "",
}
pd.concat([log, pd.DataFrame([nuevo])], ignore_index=True).to_csv(ruta_log, index=False)
print(f"✔ Registrado {nuevo['ID_Exp']} ({hash_exp})")

## Paso 6 · Llevar el modelo a la app

### Qué hace

Genera el código de Dart con la configuración de este modelo, listo para pegar en `modelos_disponibles_estructura.dart`, y si se activa copia el `.tflite` a `lib/modelos/` de la app, en la raíz del repositorio. Para que el modelo funcione igual en el teléfono todo tiene que coincidir con lo que hace la app, el espectrograma se genera con el mismo algoritmo, la división entre 255 corresponde a `Normalizacion.ceroUno`, las etiquetas van en el mismo orden alfabético de las carpetas, la neurona sigmoide corresponde a `ConfiguracionSalida.binaria` y los clips de cinco segundos coinciden con los segmentos que graba la app.

La única diferencia importante es que el entrenamiento quita los silencios y la app no, de modo que el modelo aprende con habla continua pero en el teléfono recibe cinco segundos con pausas, y si en las pruebas reales la confianza baja mucho se puede entrenar con clips que conserven el silencio, saltando la limpieza del paso 1, o quitar los silencios en la app antes de clasificar.

### Cómo se usa

Se cambia `COPIAR_A_LA_APP` a `True` si se quiere copiar el modelo, se ejecuta la celda, se pega el bloque que imprime dentro de la lista `modelosDisponibles` y se corre `flutter test` en la app, que revisa que el archivo exista y que la configuración sea válida.

In [ ]:
import shutil


# Arma el bloque de Dart con la configuración del modelo usando los parámetros de esta guía
def codigo_dart(id_modelo, nombre, version, archivo, clases):
    etiquetas = ", ".join(f"'{c}'" for c in clases)
    salida = (f"ConfiguracionSalida.binaria(etiquetas: [{etiquetas}])" if len(clases) == 2
              else f"ConfiguracionSalida.multiclase(etiquetas: [{etiquetas}])")
    return f'''  ConfiguracionModelo(
    id: '{id_modelo}',
    nombre: '{nombre}',
    version: '{version}',
    archivo: 'lib/modelos/{archivo}',
    entrada: ConfiguracionEntrada.espectrograma(
      normalizacion: Normalizacion.ceroUno,
      espectrograma: ParametrosEspectrograma(
        sampleRate: {SAMPLE_RATE},
        nFft: {N_FFT},
        hopLength: {HOP_LENGTH},
        nMels: {N_MEL},
        fMin: {F_MIN:g},
        fMax: {F_MAX:g},
        dbMin: {float(DB_MIN):g},
        mapaColor: MapaColor.inferno,
      ),
    ),
    salida: {salida},
  ),'''


# Cámbialo a True para copiar el .tflite a la carpeta de modelos de la app
COPIAR_A_LA_APP = False
if COPIAR_A_LA_APP:
    shutil.copy2(MODELO_TFLITE, APP_MODELOS / MODELO_TFLITE.name)
    print(f"✔ Copiado a {APP_MODELOS / MODELO_TFLITE.name}")

print("// Pega esto dentro de modelosDisponibles en modelos_disponibles_estructura.dart\n")
print(codigo_dart(
    id_modelo=f"{MODELO_TFLITE.stem}_{time.strftime('%Y%m%d')}",
    nombre="MobileNetV3 · " + " / ".join(c.capitalize() for c in nombres_clases),
    version="1.0",
    archivo=MODELO_TFLITE.name,
    clases=nombres_clases,
))

## Scripts anteriores, lo que se usaba antes de este proceso

### Qué son

Son scripts de versiones previas que se conservan en `Modulos no usados/cambiador` y que ya no hacen falta para entrenar, se describen aquí solo para saber qué hacía cada uno y por qué se dejó de usar.

### Qué hacían

`split_audios.py`, en `Modulos no usados/cambiador/helpers/`, cortaba los audios en clips de cinco segundos pero descartaba el último si quedaba más corto, y `PreparadorAudios.py` lo reemplazó porque rellena ese último clip con silencio y además quita los silencios. `remove_silence.py`, en la misma carpeta, quitaba los silencios con librosa usando un umbral fijo de 44 dB, y la macro `trim_silence_audacity.txt` hacía lo mismo desde Audacity, las dos se cambiaron por el umbral dinámico de `PreparadorAudios.py`, que se adapta al volumen de cada audio.

`divididor.py` repartía las imágenes en 70, 15 y 15 y balanceaba solo el conjunto de entrenamiento, y lo sustituyó `crear_split` de `entrenador.py`, que reparte en 80, 10 y 10 con los idiomas parejos desde el principio. `sas.py` reconstruía una MobileNetV3 con otra normalización y una salida softmax de dos clases para quitar las capas en float16 antes de convertir a TFLite, algo que ya no hace falta porque `entrenador.py` entrena en float32.